
# VQA-RAG Metrics Overview (branch refresh)
Quick visual report of dataset composition and model performance metrics pulled from existing CSV artifacts. Designed for side-by-side comparisons across models, datasets, and evaluation techniques.



**Data sources pulled automatically**
- `Prototyping/phase3_results/*_uc_predictions.csv` and mirrored files in `Prototyping_reformat/.../phase3_results` for per-question outputs.
- `Prototyping/phase3_results/summary_*_phase3.csv` for per-model BLEU/ROUGE/time.
- `Prototyping/scenario_outputs/*` for clinical scenario benchmarks (binary/closed/count/per-class and McNemar tests).
- `Prototyping/visualizations/*.csv` for dataset/question/answer distributions.
- `Prototyping/metadata.csv` and `Prototyping/results/colonoscopy_metadata.csv` for source/question mappings.

The notebook skips missing or empty files so it can be rerun as new results arrive. Dataset names are inferred from filenames (e.g., `*_uc_predictions` → "Ulcerative Colitis").


In [1]:

from pathlib import Path
import re

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display
from pandas.errors import EmptyDataError

sns.set_theme(style="whitegrid")
sns.set_palette("crest")
plt.rcParams["figure.figsize"] = (8, 4)
pd.set_option("display.max_colwidth", 120)


In [2]:

BASE = Path(".")

DATA = {
    "metadata": BASE / "Prototyping" / "metadata.csv",
    "colono_metadata": BASE / "Prototyping" / "results" / "colonoscopy_metadata.csv",
    "phase3_dirs": [
        BASE / "Prototyping" / "phase3_results",
        BASE / "Prototyping_reformat" / "DatasetAnalysis" / "Kvasir_VQA" / "evaluation_comparison" / "phase3_results",
    ],
    "scenario_dir": BASE / "Prototyping" / "scenario_outputs",
    "visualization_dir": BASE / "Prototyping" / "visualizations",
}

DATASET_NAME_MAP = {
    "uc": "Ulcerative Colitis",
}


def friendly_dataset(code: str) -> str:
    if not code or pd.isna(code):
        return "Unknown"
    code = str(code).lower()
    return DATASET_NAME_MAP.get(code, code.replace("_", " ").title())


def infer_dataset_from_pred_file(name: str) -> str:
    match = re.match(r"^(.*?)_(.*?)_predictions", name)
    return match.group(2) if match else "unknown"


def infer_dataset_from_summary(name: str) -> str:
    match = re.match(r"^summary_(.*?)_phase3", name)
    return match.group(1) if match else "unknown"


def load_metadata():
    meta = {}
    meta["full"] = pd.read_csv(DATA["metadata"]) if DATA["metadata"].exists() else pd.DataFrame()
    meta["colonoscopy"] = pd.read_csv(DATA["colono_metadata"]) if DATA["colono_metadata"].exists() else pd.DataFrame()
    return meta


def load_phase3_predictions():
    frames = []
    for folder in DATA["phase3_dirs"]:
        if not folder.exists():
            continue
        for csv in folder.glob("*_predictions.csv"):
            if csv.stat().st_size < 5:
                continue
            try:
                df = pd.read_csv(csv)
            except EmptyDataError:
                continue
            dataset_code = infer_dataset_from_pred_file(csv.name)
            df["dataset_code"] = dataset_code
            df["dataset"] = friendly_dataset(dataset_code)
            df["model"] = csv.stem.replace(f"_{dataset_code}_predictions", "")
            df["source_path"] = str(csv)
            frames.append(df)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True).drop_duplicates(subset=["model", "dataset", "img_id", "question"])


def load_phase3_summary():
    frames = []
    for folder in DATA["phase3_dirs"]:
        for path in folder.glob("summary_*_phase3.csv"):
            if path.stat().st_size < 5:
                continue
            try:
                df = pd.read_csv(path)
            except EmptyDataError:
                continue
            dataset_code = infer_dataset_from_summary(path.name)
            df["dataset_code"] = dataset_code
            df["dataset"] = friendly_dataset(dataset_code)
            df["source_path"] = str(path)
            frames.append(df)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True).drop_duplicates(subset=["model", "dataset"])


def load_scenario_metrics():
    folder = DATA["scenario_dir"]
    metrics = {}
    if not folder.exists():
        return metrics
    closed_path = folder / "scenario_closed_or_binary_metrics.csv"
    count_path = folder / "scenario_count_metrics.csv"
    metrics["closed"] = pd.read_csv(closed_path) if closed_path.exists() else pd.DataFrame()
    metrics["count"] = pd.read_csv(count_path) if count_path.exists() else pd.DataFrame()
    perclass_frames = []
    for csv in folder.glob("*perclass*.csv"):
        df = pd.read_csv(csv)
        df["file"] = csv.name
        perclass_frames.append(df)
    metrics["perclass"] = pd.concat(perclass_frames, ignore_index=True) if perclass_frames else pd.DataFrame()
    mcnemar_path = folder / "mcnemar_binary_viLT_vs_blipvqa.csv"
    metrics["mcnemar"] = pd.read_csv(mcnemar_path) if mcnemar_path.exists() else pd.DataFrame()
    return metrics


def load_visualization_tables():
    folder = DATA["visualization_dir"]
    def read_csv(name):
        path = folder / name
        return pd.read_csv(path) if path.exists() else pd.DataFrame()
    return {
        "question_type_counts": read_csv("question_type_counts.csv"),
        "answer_type_counts": read_csv("answer_type_counts.csv"),
        "qa_by_source": read_csv("qa_by_source.csv"),
        "crosstab_source_qtype": read_csv("crosstab_source_qtype.csv"),
        "unique_images_by_source": read_csv("unique_images_by_source.csv"),
        "top_answers": read_csv("top_nontrivial_answers.csv"),
        "top_questions": read_csv("top_question_templates.csv"),
    }


def enrich_predictions(preds: pd.DataFrame, meta: dict) -> pd.DataFrame:
    if preds.empty:
        return preds
    df = preds.copy()
    df["pred_norm"] = df.get("predicted", pd.Series(dtype=str)).fillna("").str.strip().str.lower()
    df["gt_norm"] = df.get("ground_truth", pd.Series(dtype=str)).fillna("").str.strip().str.lower()
    df["exact_match"] = df["pred_norm"] == df["gt_norm"]
    df["is_error"] = df.get("predicted", pd.Series(dtype=str)).fillna("").str.contains("ERR:", regex=False)
    if not meta.get("colonoscopy", pd.DataFrame()).empty:
        cols = [c for c in ["img_id", "question", "source", "question_type", "answer_type"] if c in meta["colonoscopy"].columns]
        df = df.merge(meta["colonoscopy"][cols], on=["img_id", "question"], how="left")
    elif not meta.get("full", pd.DataFrame()).empty:
        cols = [c for c in ["img_id", "question", "source"] if c in meta["full"].columns]
        df = df.merge(meta["full"][cols], on=["img_id", "question"], how="left")
    return df


def parse_perclass(perclass_df: pd.DataFrame) -> pd.DataFrame:
    if perclass_df.empty:
        return perclass_df
    df = perclass_df.copy()
    def parse_name(name: str):
        match = re.match(r"(.+)_perclass_(.+)\.csv", name)
        if not match:
            return None, None
        return match.group(1), match.group(2)
    df[["scenario", "model"]] = df["file"].apply(lambda x: pd.Series(parse_name(x)))
    return df


In [3]:

meta = load_metadata()
viz_tables = load_visualization_tables()
summary = load_phase3_summary()
preds_raw = load_phase3_predictions()
preds = enrich_predictions(preds_raw, meta)
scenario_metrics = load_scenario_metrics()
perclass_metrics = parse_perclass(scenario_metrics.get("perclass", pd.DataFrame()))

availability = pd.DataFrame([
    {"artifact": "metadata.csv", "rows": len(meta.get("full", pd.DataFrame()))},
    {"artifact": "colonoscopy_metadata.csv", "rows": len(meta.get("colonoscopy", pd.DataFrame()))},
    {"artifact": "phase3 summary", "rows": len(summary)},
    {"artifact": "phase3 predictions", "rows": len(preds)},
    {"artifact": "scenario closed/binary", "rows": len(scenario_metrics.get("closed", pd.DataFrame()))},
    {"artifact": "scenario count", "rows": len(scenario_metrics.get("count", pd.DataFrame()))},
    {"artifact": "scenario per-class", "rows": len(perclass_metrics)},
])

print("Artifact availability (rows):")
display(availability)
if not preds.empty:
    print("Models in predictions:", sorted(preds["model"].unique()))
    print("Datasets in predictions:", sorted(preds["dataset"].unique()))


Artifact availability (rows):


,artifact,rows
0,metadata.csv,0
1,colonoscopy_metadata.csv,0
2,phase3 summary,0
3,phase3 predictions,0
4,scenario closed/binary,0
5,scenario count,0
6,scenario per-class,0



## Dataset composition
Pulled from precomputed visualizations so it runs quickly.


In [4]:

viz = viz_tables

if not viz["qa_by_source"].empty:
    plt.figure(figsize=(8, 4))
    sns.barplot(data=viz["qa_by_source"], x="source", y="qa_count")
    plt.title("QA pairs per source")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

if not viz["unique_images_by_source"].empty:
    plt.figure(figsize=(8, 4))
    sns.barplot(data=viz["unique_images_by_source"], x="source", y="unique_images")
    plt.title("Unique images per source")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

if not viz["question_type_counts"].empty:
    plt.figure(figsize=(8, 4))
    sns.barplot(data=viz["question_type_counts"], x="question_type", y="count")
    plt.title("Question types")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

if not viz["answer_type_counts"].empty:
    plt.figure(figsize=(8, 4))
    sns.barplot(data=viz["answer_type_counts"], x="answer_type", y="count")
    plt.title("Answer types")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

if not viz["crosstab_source_qtype"].empty:
    melted = viz["crosstab_source_qtype"].melt(id_vars="source", var_name="question_type", value_name="count")
    pivot = melted.pivot(index="source", columns="question_type", values="count")
    plt.figure(figsize=(10, 6))
    sns.heatmap(pivot, annot=True, fmt=".0f", cmap="Blues")
    plt.title("Question type distribution per source")
    plt.tight_layout()
    plt.show()



## Phase 3 model metrics (by dataset)


In [5]:

if summary.empty:
    print("No summary files found.")
else:
    display(summary.drop(columns=[c for c in ["source_path"] if c in summary.columns]))
    for metric in ["BLEU_avg", "ROUGE_L", "time_per_example_sec"]:
        if metric not in summary.columns:
            continue
        g = sns.catplot(
            data=summary,
            x="model",
            y=metric,
            hue="dataset",
            kind="bar",
            height=4,
            aspect=1.6,
        )
        g.fig.suptitle(f"{metric} by model (faceted by dataset)")
        plt.xticks(rotation=15)
        plt.tight_layout()
        plt.show()


No summary files found.



## Prediction breakdown by dataset, source, and question/answer type
Uses per-question outputs to show where each model performs best/worst. Exact match is a strict lower bound; BLEU captures fuzzy overlap.


In [6]:
if preds.empty:
    print("No predictions loaded.")
else:
    preds_stats = preds.copy()
    agg_source = preds_stats.groupby(["dataset", "model", "source"], dropna=False).agg(
        mean_bleu=("bleu", "mean"),
        exact_match=("exact_match", "mean"),
        error_rate=("is_error", "mean"),
        samples=("question", "count"),
    ).reset_index()
    display(agg_source)

    g = sns.catplot(
        data=agg_source,
        x="model",
        y="mean_bleu",
        hue="source",
        col="dataset",
        kind="bar",
        height=4,
        aspect=1.4,
        sharey=False,
    )
    g.fig.suptitle("Mean BLEU by source and dataset")
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()

    if "question_type" in preds_stats.columns:
        agg_qtype = preds_stats.groupby(["dataset", "model", "question_type"], dropna=False).agg(
            mean_bleu=("bleu", "mean"),
            exact_match=("exact_match", "mean"),
            samples=("question", "count"),
        ).reset_index()
        display(agg_qtype)

        g = sns.catplot(
            data=agg_qtype,
            x="question_type",
            y="mean_bleu",
            hue="model",
            col="dataset",
            kind="bar",
            height=4,
            aspect=1.8,
            sharey=False,
        )
        g.fig.suptitle("Mean BLEU by question type and dataset")
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        plt.show()

    if "answer_type" in preds_stats.columns:
        agg_atype = preds_stats.groupby(["dataset", "model", "answer_type"], dropna=False).agg(
            mean_bleu=("bleu", "mean"),
            exact_match=("exact_match", "mean"),
            samples=("question", "count"),
        ).reset_index()
        display(agg_atype)

        g = sns.catplot(
            data=agg_atype,
            x="answer_type",
            y="mean_bleu",
            hue="model",
            col="dataset",
            kind="bar",
            height=4,
            aspect=1.8,
            sharey=False,
        )
        g.fig.suptitle("Mean BLEU by answer type and dataset")
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        plt.show()

    # Hardest/easiest questions per model within each dataset
    question_perf = preds_stats.groupby(["dataset", "model", "question"], dropna=False).agg(
        mean_bleu=("bleu", "mean"),
        exact_match=("exact_match", "mean"),
        samples=("question", "count"),
    ).reset_index()
    for dataset in sorted(preds_stats["dataset"].dropna().unique()):
        subset_ds = question_perf[question_perf["dataset"] == dataset]
        for model in sorted(subset_ds["model"].unique()):
            subset = subset_ds[subset_ds["model"] == model]
            hardest = subset.nsmallest(5, "mean_bleu")
            easiest = subset.nlargest(5, "mean_bleu")
            print(f"\n{dataset} - {model}: hardest questions")
            display(hardest)
            print(f"{dataset} - {model}: easiest questions")
            display(easiest)

No predictions loaded.



## Clinical scenario metrics (binary/closed/count/per-class)


In [7]:

closed_df = scenario_metrics.get("closed", pd.DataFrame()).copy()
if not closed_df.empty:
    for col in ["accuracy", "precision_macro", "recall_macro", "f1_macro", "cohen_kappa", "mcc"]:
        if col in closed_df.columns:
            closed_df[col] = pd.to_numeric(closed_df[col], errors="coerce")
    closed_long = closed_df.melt(id_vars=["question", "model", "labels"],
                                 value_vars=[c for c in ["accuracy", "precision_macro", "recall_macro", "f1_macro"] if c in closed_df.columns],
                                 var_name="metric", value_name="score")
    display(closed_long)
    g = sns.catplot(
        data=closed_long,
        x="question",
        y="score",
        hue="model",
        col="metric",
        kind="bar",
        col_wrap=2,
        height=4,
        sharey=False,
    )
    g.fig.suptitle("Closed/Binary scenario metrics")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No closed/binary scenario metrics.")

count_df = scenario_metrics.get("count", pd.DataFrame()).copy()
if not count_df.empty:
    for col in ["EM", "off_by_1", "MAE", "RMSE"]:
        if col in count_df.columns:
            count_df[col] = pd.to_numeric(count_df[col], errors="coerce")
    display(count_df)
    g = sns.catplot(
        data=count_df.melt(id_vars=["question", "model"], value_vars=[c for c in ["EM", "off_by_1", "MAE", "RMSE"] if c in count_df.columns], var_name="metric", value_name="score"),
        x="model",
        y="score",
        hue="question",
        col="metric",
        kind="bar",
        col_wrap=2,
        height=4,
        sharey=False,
    )
    g.fig.suptitle("Counting scenario metrics")
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()
else:
    print("No count scenario metrics.")

if not perclass_metrics.empty:
    for col in ["precision", "recall", "f1"]:
        if col in perclass_metrics.columns:
            perclass_metrics[col] = pd.to_numeric(perclass_metrics[col], errors="coerce")
    g = sns.catplot(
        data=perclass_metrics,
        x="label",
        y="f1",
        hue="model",
        col="scenario",
        kind="bar",
        col_wrap=2,
        height=4,
        sharey=False,
    )
    g.fig.suptitle("Per-class F1 by scenario")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No per-class metrics.")

mcnemar_df = scenario_metrics.get("mcnemar", pd.DataFrame())
if not mcnemar_df.empty:
    display(mcnemar_df)


No closed/binary scenario metrics.
No count scenario metrics.
No per-class metrics.



## Refresh or extend
- Drop new prediction CSVs or summaries into the existing folders and rerun the notebook; loaders skip empty files.
- To add another dataset, follow the filename pattern (`*_DATASET_predictions.csv`, `summary_DATASET_phase3.csv`); the dataset name will be parsed automatically.
- If you generate new scenario evaluations, save them into `Prototyping/scenario_outputs` with the existing naming scheme so the charts pick them up automatically.
